# GameTheory-16c (Rust) : La dimension paiement que le designer n'a jamais cherchée — extraction de revenu sous contraintes locales au premier meilleur

**Objectifs** : énumérer les paiements, comparer les frontières et expliciter les contraintes du jouet.

**Prérequis** : mécanismes binaires, utilité quasi-linéaire ; **durée** : 75 minutes.

**Navigation** : [Version Python](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) | [Version C#](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-CSharp.ipynb) | [<< 16b-Automated-Mechanism-Design](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) | [Index GameTheory](README.md) | [16k-Encheres-Myerson >>](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb)

GT-16b construit un générateur de tables, un vérificateur indépendant et un témoin d'impossibilité. Le générateur ne cherche que 16 tables d'issues et fixe les paiements à zéro. Ici l'énumération couvre les 4096 mécanismes : sur l'instance (1,1), 47 satisfont les contraintes locales, dont 34 au bien-être maximal et 29 avec un revenu positif. Le générateur atteint bien J=2 et un revenu nul, mais sa table échoue au test local de non-déviation ; la sortie montre la déviation profitable.

Deux variantes de participation sont comparées : l'IR à types fixés, appelée point-prior dans ce jouet, et l'IR uniforme sur tous les types et tous les reports. La seconde force ici les paiements à zéro, alors que la première permet l'extraction complète sur (1,1).

**Portée des vérificateurs.** La non-déviation locale fixe les deux types et compare chaque changement unilatéral depuis chaque profil de reports. Comme les deux sens d'une même arête sont contrôlés, elle impose l'égalité des utilités entre reports propres. Le DSIC usuel compare, pour chaque type et chaque report des autres, le report sincère à ses déviations ; ce n'est pas le test implémenté ici. L'IR uniforme ne calcule aucune espérance et n'est pas l'IR ex-interim usuelle.

La référence pédagogique primaire est le source Python versionné cité en fin de carnet. Les publications historiques qui y figuraient sont conservées comme contexte, sans attribuer les résultats de ce jouet à leurs théorèmes.

## Objectifs d'apprentissage

1. Lire les 16 candidats du générateur et son choix de paiements nuls.
2. Énumérer les 4096 mécanismes et mesurer la frontière revenu/bien-être sur les quatre instances à types fixés.
3. Faire vérifier le générateur et le mécanisme extracteur par le même organe indépendant.
4. Distinguer l'IR à types fixés de l'IR uniforme sur types et reports.
5. Borner les conclusions au modèle exécuté, sans confondre sa non-déviation locale avec le DSIC usuel.

## 1. Ce que le designer committé designe vraiment

Le domaine de GT-16b : 2 agents, types θ ∈ {0,1}, issue o ∈ {0,1}, paiements ∈ {0,1}². Un mécanisme est un couple de tables : `issue(r) ∈ {0,1}` et `paiement(r) ∈ {0,1}²` pour chacun des 4 profils de reports `r`. L'espace COMPLET des mécanismes compte `2⁴ × 4⁴ = 4096` candidats. Le générateur de GT-16b n'en visite que `2⁴ = 16` : il énumère les issues, prend l'argmax de bien-être, puis **fixe tous les paiements à zéro** (« les paiements minimaux qui satisfont trivialement IR », dit son commentaire). Le choix est défendable — mais c'est un choix : l'utilité quasi-linéaire `uᵢ = θᵢ·issue − paiementᵢ` rend les transferts neutres pour le bien-être `J = Σθᵢ·issue`, donc ignorer les paiements ne coûte rien au critère optimisé… et coûte tout au seul autre critère naturel, le revenu.

Reproduisons d'abord le designer committé sur l'instance `true = (1,1)` — le point-prior où les deux agents sont de type 1.

In [1]:
use std::collections::BTreeMap;

// Runtime affiche par le port : jamais le mot « Python » (tolerance de presence).
println!("Runtime Rust : std seul (evcxr)");

const PROFILES: [(i32, i32); 4] = [(0, 0), (0, 1), (1, 0), (1, 1)];   // profils de reports (2 agents, 2 types)
const N_AGENTS: usize = 2;
#[allow(dead_code)]
const PAY_RANGE: [i32; 2] = [0, 1];

/// Repr Python d'un tuple d'entiers.
fn py_tuple(t: (i32, i32)) -> String { format!("({}, {})", t.0, t.1) }

/// Reproduction du designer GT-16b : issues par argmax, paiements ~ 0.
fn generate_gt16b(true_types: (i32, i32)) -> (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>, i32) {
    let tt = [true_types.0, true_types.1];
    let mut best_j: i32 = -1;
    let mut best_issue: Vec<i32> = Vec::new();
    // Les 16 tables d'issues, dans l'ordre lexicographique de itertools.product :
    // le profil 0 porte le bit de poids fort.
    for n in 0..(1usize << PROFILES.len()) {
        let mut issue_choice: Vec<i32> = Vec::new();
        for k in 0..PROFILES.len() {
            issue_choice.push(((n >> (PROFILES.len() - 1 - k)) & 1) as i32);
        }
        let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            issue_table.insert(*p, issue_choice[k]);
        }
        // J : la somme porte sur les agents, la lecture sur le profil vrai.
        let j: i32 = (0..N_AGENTS).map(|i| tt[i] * issue_table[&(tt[0], tt[1])]).sum();
        // `>` strict : la premiere table a J maximal est conservee.
        if j > best_j {
            best_j = j;
            best_issue = issue_choice;
        }
    }
    let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
    for (k, p) in PROFILES.iter().enumerate() {
        issue_table.insert(*p, best_issue[k]);
    }
    let mut payment_table: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
    for p in PROFILES.iter() {
        payment_table.insert(*p, (0, 0));
    }
    (issue_table, payment_table, best_j)
}

const TRUE_11: (i32, i32) = (1, 1);
let (issue_16b, pay_16b, j_16b) = generate_gt16b(TRUE_11);
let revenue_16b: i32 = pay_16b[&TRUE_11].0 + pay_16b[&TRUE_11].1;
println!("Instance point-prior : true = {}", py_tuple(TRUE_11));
println!("Designer GT-16b -> issue((1,1)) = {}, paiements ~ 0 partout", issue_16b[&(1, 1)]);
println!("Bien-etre J = {} (premier meilleur)   Revenu = {}", j_16b, revenue_16b);
println!();
println!("Espace visite par le generateur : {} tables d'issues", 2usize.pow(PROFILES.len() as u32));
println!("Espace complet des mecanismes   : {} x {} = {} candidats",
    2usize.pow(PROFILES.len() as u32),
    4usize.pow(PROFILES.len() as u32),
    2usize.pow(PROFILES.len() as u32) * 4usize.pow(PROFILES.len() as u32));

Runtime Rust : std seul (evcxr)


Instance point-prior : true = (1, 1)


Designer GT-16b -> issue((1,1)) = 1, paiements ~ 0 partout


Bien-etre J = 2 (premier meilleur)   Revenu = 0


Espace visite par le generateur : 16 tables d'issues


Espace complet des mecanismes   : 16 x 256 = 4096 candidats


**Lecture de la sortie.** Le générateur atteint le bien-être maximal J=2 sur (1,1), avec un revenu nul. Il visite 16 tables d'issues sur 4096 mécanismes, soit environ 0,4% de l'espace. Cela ne prouve pas l'admissibilité : la section suivante soumet réellement sa table au vérificateur local et constate son échec. L'énumération conjointe sépare ainsi trois questions : bien-être, revenu et respect des contraintes.

## 2. L'énumération conjointe — le vérificateur hérité tranche

On hérite l'organe de GT-16b : son **vérificateur indépendant**, en copie locale, zéro import du générateur — mêmes définitions de l'utilité quasi-linéaire, de non-déviation locale (aucune déviation unilatérale strictement profitable, pour chaque agent et chaque profil reporté) et d'IR (utilité ≥ 0 pour chaque agent et chaque profil, **aux types vrais** — la sémantique exacte de `verify_IR` de GT-16b). Puis on énumère les 4096 candidats de l'espace complet, et on compte par instance point-prior : les admissibles (non-déviation locale ∧ IR), parmi eux les à bien-être optimal, parmi ceux-ci les à revenu strictement positif, et le revenu maximal.

In [2]:
use std::collections::BTreeMap;

type Mec = (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>);

const PROFILES: [(i32, i32); 4] = [(0, 0), (0, 1), (1, 0), (1, 1)];   // profils de reports (2 agents, 2 types)
const N_AGENTS: usize = 2;
const PAY_RANGE: [i32; 2] = [0, 1];
const TRUE_11: (i32, i32) = (1, 1);

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un tuple d'entiers.
fn py_tuple(t: (i32, i32)) -> String { format!("({}, {})", t.0, t.1) }
/// Repr Python d'un couple (booleen, message) : apostrophes simples.
fn py_pair(p: (bool, String)) -> String { format!("({}, '{}')", py_bool(p.0), p.1) }
/// Lecture du i-eme paiement d'un couple.
fn pay_at(p: (i32, i32), i: usize) -> i32 { if i == 0 { p.0 } else { p.1 } }
/// Lecture du i-eme type d'un couple.
fn theta_at(t: (i32, i32), i: usize) -> i32 { if i == 0 { t.0 } else { t.1 } }

/// Reproduction du designer GT-16b : issues par argmax, paiements ~ 0.
fn generate_gt16b(true_types: (i32, i32)) -> (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>, i32) {
    let tt = [true_types.0, true_types.1];
    let mut best_j: i32 = -1;
    let mut best_issue: Vec<i32> = Vec::new();
    // Les 16 tables d'issues, dans l'ordre lexicographique de itertools.product :
    // le profil 0 porte le bit de poids fort.
    for n in 0..(1usize << PROFILES.len()) {
        let mut issue_choice: Vec<i32> = Vec::new();
        for k in 0..PROFILES.len() {
            issue_choice.push(((n >> (PROFILES.len() - 1 - k)) & 1) as i32);
        }
        let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            issue_table.insert(*p, issue_choice[k]);
        }
        // J : la somme porte sur les agents, la lecture sur le profil vrai.
        let j: i32 = (0..N_AGENTS).map(|i| tt[i] * issue_table[&(tt[0], tt[1])]).sum();
        // `>` strict : la premiere table a J maximal est conservee.
        if j > best_j {
            best_j = j;
            best_issue = issue_choice;
        }
    }
    let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
    for (k, p) in PROFILES.iter().enumerate() {
        issue_table.insert(*p, best_issue[k]);
    }
    let mut payment_table: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
    for p in PROFILES.iter() {
        payment_table.insert(*p, (0, 0));
    }
    (issue_table, payment_table, best_j)
}

/// Utilite grossiere (hors paiement) — copie locale de verify_utility_i.
fn verif_utility_i(theta_i: i32, issue: i32) -> i32 { theta_i * issue }

/// Aucune deviation unilaterale strictement profitable, chaque agent, chaque profil.
fn verif_dsic(m: &Mec, true_types: (i32, i32)) -> (bool, String) {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            let u_sincere = verif_utility_i(theta_at(true_types, i), issue_table[r]) - pay_at(payment_table[r], i);
            for theta_dev in [0, 1].iter() {
                let mut r_dev = *r;
                if i == 0 { r_dev.0 = *theta_dev; } else { r_dev.1 = *theta_dev; }
                let u_dev = verif_utility_i(theta_at(true_types, i), issue_table[&r_dev]) - pay_at(payment_table[&r_dev], i);
                // Domaine entier : `u_dev > u_sincere + 1e-9` equivaut a `u_dev > u_sincere`.
                if u_dev > u_sincere {
                    return (false, format!("Non-déviation locale violée : agent {} prefere {} a {}", i, py_tuple(r_dev), py_tuple(*r)));
                }
            }
        }
    }
    (true, "non-déviation locale OK".to_string())
}

/// Utilite >= 0 pour chaque agent et chaque profil, AUX TYPES VRAIS (semantique GT-16b).
fn verif_ir(m: &Mec, true_types: (i32, i32)) -> (bool, String) {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            let theta_i = theta_at(true_types, i);
            let u = verif_utility_i(theta_i, issue_table[r]) - pay_at(payment_table[r], i);
            if u < 0 {
                return (false, format!("IR fail: agent {} u={} en profil {}", i, u, py_tuple(*r)));
            }
        }
    }
    (true, "IR OK".to_string())
}

/// Bien-etre au profil vrai.
fn verif_welfare(m: &Mec, true_types: (i32, i32)) -> i32 {
    let (issue_table, _) = m;
    (0..N_AGENTS).map(|i| verif_utility_i(theta_at(true_types, i), issue_table[&true_types])).sum()
}

/// Revenu collecte au profil vrai (non-deviation locale => report sincere).
fn verif_revenue(m: &Mec, true_types: (i32, i32)) -> i32 {
    let (_, payment_table) = m;
    pay_at(payment_table[&true_types], 0) + pay_at(payment_table[&true_types], 1)
}

/// Espace COMPLET : 4096 candidats (issue, paiement).
fn enumerer_tous() -> Vec<Mec> {
    let mut out: Vec<Mec> = Vec::new();
    let n_profiles = PROFILES.len();
    for n in 0..(1usize << n_profiles) * (1usize << (2 * n_profiles)) {
        let issue_code = n >> (2 * n_profiles);
        let pay_code = n & ((1usize << (2 * n_profiles)) - 1);
        let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            let bit = (issue_code >> (n_profiles - 1 - k)) & 1;
            issue_table.insert(*p, bit as i32);
        }
        let mut payment_table: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            let b0 = (pay_code >> (2 * (n_profiles - 1 - k) + 1)) & 1;
            let b1 = (pay_code >> (2 * (n_profiles - 1 - k))) & 1;
            payment_table.insert(*p, (PAY_RANGE[b0 as usize], PAY_RANGE[b1 as usize]));
        }
        out.push((issue_table, payment_table));
    }
    out
}

// === Verificateur herite de GT-16b (copie locale, zero import du generateur) ===
let (issue_16b, pay_16b, _) = generate_gt16b(TRUE_11);
let cands = enumerer_tous();
println!("{:>10} {:>12} {:>12} {:>14} {:>11}", "instance", "admissibles", "welfare-opt", "dont revenu>0", "max rev@fb");
println!("{}", "-".repeat(66));
for true_types in PROFILES.iter().copied() {
    let fb: i32 = true_types.0 + true_types.1;
    let mut n_adm = 0i32;
    let mut n_fb = 0i32;
    let mut n_pos = 0i32;
    let mut best_rev = -1i32;
    for m in cands.iter() {
        if verif_dsic(m, true_types).0 && verif_ir(m, true_types).0 {
            n_adm += 1;
            if verif_welfare(m, true_types) == fb {
                n_fb += 1;
                let rev = verif_revenue(m, true_types);
                if rev > 0 { n_pos += 1; }
                if rev > best_rev { best_rev = rev; }
            }
        }
    }
    println!("{:>10} {:>12} {:>12} {:>14} {:>11}", py_tuple(true_types), n_adm, n_fb, n_pos, best_rev);
}
let mec_16b: Mec = (issue_16b, pay_16b);
println!("Générateur hérité, non-déviation locale : {} ; IR : {}",
    py_pair(verif_dsic(&mec_16b, TRUE_11)), py_pair(verif_ir(&mec_16b, TRUE_11)));

  instance  admissibles  welfare-opt  dont revenu>0  max rev@fb


------------------------------------------------------------------


    (0, 0)           16           16              0           0


    (0, 1)           25           15             10           1


    (1, 0)           25           15             10           1


    (1, 1)           47           34             29           2


Générateur hérité, non-déviation locale : (False, 'Non-déviation locale violée : agent 0 prefere (1, 1) a (0, 1)') ; IR : (True, 'IR OK')


**Lecture de la sortie.** Dans l'ordre(0,0),(0,1),(1,0),(1,1), les nombres de mécanismes admissibles sont 16, 25, 25, 47. Sur(1,1), 34 atteignent le bien-être maximal ; 29 de ceux-ci ont un revenu positif et le maximum vaut 2. Le générateur hérité atteint lui aussi J=2, mais sa table viole la non-déviation locale : à types (1,1), l'agent 0 préfère le report (1,1) au report (0,1), pour une utilité 1 au lieu de 0. Son revenu nul et son défaut d'admissibilité sont deux constats distincts.

Sur(0,0), l'IR et les paiements non négatifs imposent des paiements nuls : les 16 tables d'issues sont admissibles et aucun revenu positif n'existe. Le revenu maximal au premier meilleur vaut respectivement0,1,1,2 sur les quatre instances, soit la somme des types dans ce modèle.

## Exercice 1 : l'instance (0,0) est une forteresse

La table de la section 2 montre 16 mécanismes admissibles sur `(0,0)` et aucun à revenu positif. Démontrez-le proprement : pour chaque agent de type 0, écrivez la contrainte IR point-prior et montrez qu'elle force `paiementᵢ(r) = 0` pour tout profil `r` — indépendamment de l'issue. Concluez : le revenu maximal d'une instance est nul dès que tous les types sont nuls, et la forme close `Σθᵢ` est cohérente. Vérifiez enfin votre preuve par énumération sur les deux autres instances à agent de type 0 : leur revenu ne peut venir que de l'agent intéressé.

Complétez `borne_revenu` (Python) ou `borneRevenu` (C#), pour deux types binaires et une issue binaire ; retournez la borne totale de revenu impliquée par IR. Le garde mélange cas nuls et non nuls, puis votre preuve explique pourquoi un type nul ne finance rien.

In [3]:
// Exercice 1 a completer
// 1. Pour true = (0,0) : ecrire la contrainte IR d'un agent de type 0 (u_i = 0*issue - p_i)
// 2. En deduire p_i(r) = 0 pour tout r, puis revenu = 0 pour TOUT mecanisme admissible
// 3. Verifier : FRONTIERE[(0,0)][2] == 0 et FRONTIERE[(0,0)][3] == 0 ; puis (0,1) : revenu <= 1
// Indice : u_i = -p_i(r) >= 0 et p_i(r) dans {0,1} => p_i(r) = 0
fn borne_revenu(_type0: i32, _type1: i32, _issue: i32) -> Option<f64> {
    // TODO etudiant : borne du revenu impliquee par l'IR
    None
}

let cas_borne: [(i32, i32, i32, f64); 6] = [
    (0, 0, 0, 0.0), (0, 0, 1, 0.0), (0, 1, 1, 1.0),
    (1, 0, 1, 1.0), (1, 1, 1, 2.0), (1, 1, 0, 0.0),
];
let rep_borne: Vec<Option<f64>> = cas_borne.iter().map(|c| borne_revenu(c.0, c.1, c.2)).collect();
if rep_borne.iter().all(|x| x.is_none()) {
    println!("Exercice 1 à compléter : borne IR et preuve pour les types nuls");
} else {
    // Branche de validation : aucun panic, on n'imprime que si tout passe.
    let tout_ok = rep_borne.iter().zip(cas_borne.iter()).all(|(x, c)| match x {
        Some(v) => v.is_finite() && (v - c.3).abs() < 1e-9,
        None => false,
    });
    if tout_ok {
        println!("Exercice 1 : bornes validées ; rédigez la preuve et relisez la frontière");
    }
};
// Le point-virgule final supprime le `execute_result ()` qu'evcxr publie
// quand la derniere instruction d'une cellule est un bloc (`if`/`else`).

Exercice 1 à compléter : borne IR et preuve pour les types nuls


## 3. Le mécanisme extrateur — exhibé et vérifié par l'organe hérité

Sur l'instance (1,1), l'extracteur produit l'issue 1 et facture 1 à chaque agent au report (1,1) ; les autres profils donnent l'issue 0 et des paiements nuls. Pour les types fixés à 1, chaque agent a une utilité nulle à tous les reports. La non-déviation locale est donc satisfaite par égalité, l'IR est serrée, le bien-être vaut 2 et le revenu vaut 2. La cellule suivante fait réellement ces vérifications.

Cette propriété concerne les types (1,1). Elle ne donne aucune garantie d'incitation ou de participation pour d'autres types sous la définition habituelle de DSIC.

In [4]:
use std::collections::BTreeMap;

type Mec = (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>);

const PROFILES: [(i32, i32); 4] = [(0, 0), (0, 1), (1, 0), (1, 1)];   // profils de reports (2 agents, 2 types)
const N_AGENTS: usize = 2;
const TRUE_11: (i32, i32) = (1, 1);

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un tuple d'entiers.
/// Lecture du i-eme paiement d'un couple.
fn pay_at(p: (i32, i32), i: usize) -> i32 { if i == 0 { p.0 } else { p.1 } }
/// Lecture du i-eme type d'un couple.
fn theta_at(t: (i32, i32), i: usize) -> i32 { if i == 0 { t.0 } else { t.1 } }

/// Utilite grossiere (hors paiement) — copie locale de verify_utility_i.
fn verif_utility_i(theta_i: i32, issue: i32) -> i32 { theta_i * issue }

/// Aucune deviation unilaterale strictement profitable, chaque agent, chaque profil.
fn verif_dsic(m: &Mec, true_types: (i32, i32)) -> (bool, String) {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            let u_sincere = verif_utility_i(theta_at(true_types, i), issue_table[r]) - pay_at(payment_table[r], i);
            for theta_dev in [0, 1].iter() {
                let mut r_dev = *r;
                if i == 0 { r_dev.0 = *theta_dev; } else { r_dev.1 = *theta_dev; }
                let u_dev = verif_utility_i(theta_at(true_types, i), issue_table[&r_dev]) - pay_at(payment_table[&r_dev], i);
                // Domaine entier : `u_dev > u_sincere + 1e-9` equivaut a `u_dev > u_sincere`.
                if u_dev > u_sincere {
                    return (false, format!("Non-déviation locale violée : agent {} prefere {} a {}", i, py_tuple(r_dev), py_tuple(*r)));
                }
            }
        }
    }
    (true, "non-déviation locale OK".to_string())
}

/// Utilite >= 0 pour chaque agent et chaque profil, AUX TYPES VRAIS (semantique GT-16b).
fn verif_ir(m: &Mec, true_types: (i32, i32)) -> (bool, String) {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            let theta_i = theta_at(true_types, i);
            let u = verif_utility_i(theta_i, issue_table[r]) - pay_at(payment_table[r], i);
            if u < 0 {
                return (false, format!("IR fail: agent {} u={} en profil {}", i, u, py_tuple(*r)));
            }
        }
    }
    (true, "IR OK".to_string())
}

/// Bien-etre au profil vrai.
fn verif_welfare(m: &Mec, true_types: (i32, i32)) -> i32 {
    let (issue_table, _) = m;
    (0..N_AGENTS).map(|i| verif_utility_i(theta_at(true_types, i), issue_table[&true_types])).sum()
}

/// Revenu collecte au profil vrai (non-deviation locale => report sincere).
fn verif_revenue(m: &Mec, true_types: (i32, i32)) -> i32 {
    let (_, payment_table) = m;
    pay_at(payment_table[&true_types], 0) + pay_at(payment_table[&true_types], 1)
}

// === Le mecanisme extracteur, construit puis VERIFIE par l'organe herite ===
let m_extracteur: Mec = (
    BTreeMap::from([((0, 0), 0), ((0, 1), 0), ((1, 0), 0), ((1, 1), 1)]),
    BTreeMap::from([((0, 0), (0, 0)), ((0, 1), (0, 0)), ((1, 0), (0, 0)), ((1, 1), (1, 1))]),
);

let (ok_dsic, msg_dsic) = verif_dsic(&m_extracteur, TRUE_11);
let (ok_ir, msg_ir) = verif_ir(&m_extracteur, TRUE_11);
println!("Mechanisme extracteur sur true = (1,1), verifie par le meme organe que GT-16b :");
println!("  non-déviation locale : {}  ({})", py_bool(ok_dsic), msg_dsic);
println!("  IR   : {}  ({})", py_bool(ok_ir), msg_ir);
println!("  Bien-etre : {}  (premier meilleur = {})", verif_welfare(&m_extracteur, TRUE_11), TRUE_11.0 + TRUE_11.1);
println!("  Revenu    : {}  (max de la frontiere)", verif_revenue(&m_extracteur, TRUE_11));
println!();
// Le detail non-deviation locale : pourquoi aucune deviation n'est strictement profitable
// (acces direct aux tables : evcxr ne persiste pas une reference non-'static).
println!("Pourquoi non-déviation locale tient (agent 1, profil sincere (1,1)) :");
let u_sincere = verif_utility_i(1, m_extracteur.0[&(1, 1)]) - pay_at(m_extracteur.1[&(1, 1)], 0);
let u_dev = verif_utility_i(1, m_extracteur.0[&(0, 1)]) - pay_at(m_extracteur.1[&(0, 1)], 0);
println!("  sincere        : u = 1*{} - {} = {}", m_extracteur.0[&(1, 1)], pay_at(m_extracteur.1[&(1, 1)], 0), u_sincere);
println!("  devier (->0)   : u = 1*{} - {} = {}  (jamais > sincere)", m_extracteur.0[&(0, 1)], pay_at(m_extracteur.1[&(0, 1)], 0), u_dev);
println!();
println!("Contre-temoin au designer GT-16b : bien-etre identique ({} = J du designer),", verif_welfare(&m_extracteur, TRUE_11));
println!("revenu {} vs 0 : l'effondrement paiements ~ 0 coutait 100% du revenu.", verif_revenue(&m_extracteur, TRUE_11));

Mechanisme extracteur sur true = (1,1), verifie par le meme organe que GT-16b :


  non-déviation locale : True  (non-déviation locale OK)


  IR   : True  (IR OK)


  Bien-etre : 2  (premier meilleur = 2)


  Revenu    : 2  (max de la frontiere)


Pourquoi non-déviation locale tient (agent 1, profil sincere (1,1)) :


  sincere        : u = 1*1 - 1 = 0


  devier (->0)   : u = 1*0 - 0 = 0  (jamais > sincere)


Contre-temoin au designer GT-16b : bien-etre identique (2 = J du designer),


revenu 2 vs 0 : l'effondrement paiements ~ 0 coutait 100% du revenu.


**Lecture de la sortie.** L'extracteur passe la non-déviation locale et l'IR sur les types (1,1), avec bien-être 2 et revenu 2. L'exemple de changement de report donne une utilité 0 de part et d'autre ; le vérificateur contrôle également les autres profils. Le designer connaît les types pour cette expérience : il s'agit d'information complète de son point de vue. Aucun résultat d'extraction à prior distribué n'est démontré ou reproduit.

## Exercice 2 : élargir la gamme des paiements

Toute la -c utilise des paiements ∈ {0, 1}. Sur l'instance `(1,1)`, élargissez à {0, 1, 2} et relancez l'énumération conjointe (l'espace croît à `2⁴ × 9⁴` — reste trivial). Écrivez d'abord la borne : IR point-prior pour un agent de type 1 exige `paiementᵢ(r) ≤ issue(r) ≤ 1` — le revenu au profil vrai est donc plafonné à 2, quelle que soit la gamme. Confirmez par l'énumération, puis observez ce que la gamme étendue ne change PAS : le bien-être `J` ne compte pas les paiements — dites ce que ça révèle de la définition du bien-être du jouet (un transfert de l'agent au designer y est invisible : ni coût, ni benefice social).

Complétez `mesurer_frontiere` / `mesurerFrontiere` : consommez l’itérable des mécanismes fourni et retournez le quadruplet (admissibles, premier meilleur, revenu positif, revenu maximal). Le garde observe le nombre de candidats réellement parcourus, dont les 104 976 candidats de la gamme étendue.

In [5]:
use std::collections::BTreeMap;

type Mec = (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>);

const PROFILES: [(i32, i32); 4] = [(0, 0), (0, 1), (1, 0), (1, 1)];   // profils de reports (2 agents, 2 types)
const N_AGENTS: usize = 2;

// Exercice 2 a completer
// 1. Prediction ecrite AVANT : max revenu @ fb sur (1,1) avec paiements dans {0,1,2} = ?
// 2. Enumerer (issues 2^4) x (paiements {0,1,2}^(2*4)) avec le meme verificateur
// 3. Comparer au max = 2 : la borne vient de IR (p_i <= issue <= 1), pas de la gamme
// 4. Bonus : J change-t-il avec les paiements ? (J ne compte que les utilites grossieres)
// Indice : pour l'agent i de type 1, IR en r exige 1*issue(r) - p_i(r) >= 0, et issue(r) <= 1
fn mesurer_frontiere<I>(_candidats: I, _type0: i32, _type1: i32) -> Option<(i32, i32, i32, i32)>
where I: Iterator<Item = Mec> {
    // TODO etudiant : consommer l'enumeration et renvoyer quatre statistiques
    None
}

// La generation des candidats (16 x (max_paiement+1)**8, chiffres en base max_paiement+1,
// issues puis paiements dans l'ordre itertools.product, garde compteur == 16*(max_paiement+1)**8,
// soit 104 976 a max_paiement = 2) est l'objet de l'exercice : le stub n'en consomme aucun.
let cas_frontiere: [(i32, i32, i32); 6] = [(1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1), (2, 1, 1), (2, 0, 1)];
let mut completes = 0;
for (max_paiement, t0, t1) in cas_frontiere.iter().copied() {
    let reponse = mesurer_frontiere(std::iter::empty::<Mec>(), t0, t1);
    if reponse.is_none() {
        continue;
    }
    completes += 1;
    let _ = max_paiement;
}
if completes == 0 {
    println!("Exercice 2 à compléter : gamme des paiements étendue");
} else {
    println!("Exercice 2 : six frontières et budgets d'énumération validés");
};
// Le point-virgule final supprime le `execute_result ()` qu'evcxr publie
// quand la derniere instruction d'une cellule est un bloc (`if`/`else`).

Exercice 2 à compléter : gamme des paiements étendue


## 4. La couture du témoin d'impossibilité — deux lectures d'IR qui ne se rencontrent pas

Le témoin hérité interdit les paiements au moins égaux à 1 à tous les reports lorsqu'un agent de type 0 doit participer : son utilité serait alors au plus −1. L'IR à types fixés applique cette contrainte seulement aux types présents. Sur(1,1), aucun type nul n'est présent ; l'énumération doit donc vérifier si un paiement uniforme survit.

- **IR à types fixés (point-prior dans le jouet)** : pour chaque profil de reports et chaque agent, l'utilité de son type fixé doit être positive ou nulle.
- **IR uniforme sur types et reports** : pour chaque profil, agent et type possible, l'utilité doit être positive ou nulle. C'est une contrainte robuste, y compris quand type et report diffèrent.

La seconde n'est pas l'IR ex-interim usuelle : aucune espérance conditionnelle sur les types des autres agents n'est calculée. On conserve les deux expériences et on mesure leurs conséquences séparément.

In [6]:
use std::collections::{BTreeMap, BTreeSet};

type Mec = (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>);

const PROFILES: [(i32, i32); 4] = [(0, 0), (0, 1), (1, 0), (1, 1)];   // profils de reports (2 agents, 2 types)
const N_AGENTS: usize = 2;
const PAY_RANGE: [i32; 2] = [0, 1];
const TRUE_11: (i32, i32) = (1, 1);

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un tuple d'entiers.
fn py_tuple(t: (i32, i32)) -> String { format!("({}, {})", t.0, t.1) }
/// Lecture du i-eme paiement d'un couple.
fn pay_at(p: (i32, i32), i: usize) -> i32 { if i == 0 { p.0 } else { p.1 } }
/// Lecture du i-eme type d'un couple.
fn theta_at(t: (i32, i32), i: usize) -> i32 { if i == 0 { t.0 } else { t.1 } }

/// Utilite grossiere (hors paiement) — copie locale de verify_utility_i.
fn verif_utility_i(theta_i: i32, issue: i32) -> i32 { theta_i * issue }

/// Aucune deviation unilaterale strictement profitable, chaque agent, chaque profil.
fn verif_dsic(m: &Mec, true_types: (i32, i32)) -> (bool, String) {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            let u_sincere = verif_utility_i(theta_at(true_types, i), issue_table[r]) - pay_at(payment_table[r], i);
            for theta_dev in [0, 1].iter() {
                let mut r_dev = *r;
                if i == 0 { r_dev.0 = *theta_dev; } else { r_dev.1 = *theta_dev; }
                let u_dev = verif_utility_i(theta_at(true_types, i), issue_table[&r_dev]) - pay_at(payment_table[&r_dev], i);
                // Domaine entier : `u_dev > u_sincere + 1e-9` equivaut a `u_dev > u_sincere`.
                if u_dev > u_sincere {
                    return (false, format!("Non-déviation locale violée : agent {} prefere {} a {}", i, py_tuple(r_dev), py_tuple(*r)));
                }
            }
        }
    }
    (true, "non-déviation locale OK".to_string())
}

/// Utilite >= 0 pour chaque agent et chaque profil, AUX TYPES VRAIS (semantique GT-16b).
fn verif_ir(m: &Mec, true_types: (i32, i32)) -> (bool, String) {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            let theta_i = theta_at(true_types, i);
            let u = verif_utility_i(theta_i, issue_table[r]) - pay_at(payment_table[r], i);
            if u < 0 {
                return (false, format!("IR fail: agent {} u={} en profil {}", i, u, py_tuple(*r)));
            }
        }
    }
    (true, "IR OK".to_string())
}

/// Bien-etre au profil vrai.
fn verif_welfare(m: &Mec, true_types: (i32, i32)) -> i32 {
    let (issue_table, _) = m;
    (0..N_AGENTS).map(|i| verif_utility_i(theta_at(true_types, i), issue_table[&true_types])).sum()
}

/// Revenu collecte au profil vrai (non-deviation locale => report sincere).
fn verif_revenue(m: &Mec, true_types: (i32, i32)) -> i32 {
    let (_, payment_table) = m;
    pay_at(payment_table[&true_types], 0) + pay_at(payment_table[&true_types], 1)
}

/// Espace COMPLET : 4096 candidats (issue, paiement).
fn enumerer_tous() -> Vec<Mec> {
    let mut out: Vec<Mec> = Vec::new();
    let n_profiles = PROFILES.len();
    for n in 0..(1usize << n_profiles) * (1usize << (2 * n_profiles)) {
        let issue_code = n >> (2 * n_profiles);
        let pay_code = n & ((1usize << (2 * n_profiles)) - 1);
        let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            let bit = (issue_code >> (n_profiles - 1 - k)) & 1;
            issue_table.insert(*p, bit as i32);
        }
        let mut payment_table: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            let b0 = (pay_code >> (2 * (n_profiles - 1 - k) + 1)) & 1;
            let b1 = (pay_code >> (2 * (n_profiles - 1 - k))) & 1;
            payment_table.insert(*p, (PAY_RANGE[b0 as usize], PAY_RANGE[b1 as usize]));
        }
        out.push((issue_table, payment_table));
    }
    out
}

/// Candidats avec paiement >= 1 PARTOUT (la contrainte du temoin GT-16b).
fn enumerer_uniformes() -> Vec<Mec> {
    let mut out: Vec<Mec> = Vec::new();
    let n_profiles = PROFILES.len();
    for n in 0..(1usize << n_profiles) * (1usize << (2 * n_profiles)) {
        let issue_code = n >> (2 * n_profiles);
        let pay_code = n & ((1usize << (2 * n_profiles)) - 1);
        let mut issue_table: BTreeMap<(i32, i32), i32> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            let bit = (issue_code >> (n_profiles - 1 - k)) & 1;
            issue_table.insert(*p, bit as i32);
        }
        let mut payment_table: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
        for (k, p) in PROFILES.iter().enumerate() {
            let b0 = (pay_code >> (2 * (n_profiles - 1 - k) + 1)) & 1;
            let b1 = (pay_code >> (2 * (n_profiles - 1 - k))) & 1;
            payment_table.insert(*p, (1 + b0 as i32, 1 + b1 as i32));
        }
        out.push((issue_table, payment_table));
    }
    out
}

/// Tout type possible pourrait se presenter : u >= 0 pour chaque theta de chaque agent.
fn verif_ir_uniforme(m: &Mec) -> bool {
    let (issue_table, payment_table) = m;
    for r in PROFILES.iter() {
        for i in 0..N_AGENTS {
            for theta in [0, 1].iter() {
                if verif_utility_i(*theta, issue_table[r]) - pay_at(payment_table[r], i) < 0 {
                    return false;
                }
            }
        }
    }
    true
}

// === Cote 1 : le temoin, reproduit la ou son agent de type 0 existe ===
println!("Temoin GT-16b (paiement uniforme >= 1), IR point-prior, instance par instance :");
let uniformes = enumerer_uniformes();
for true_types in PROFILES.iter().copied() {
    let mut n = 0i32;
    for m in uniformes.iter() {
        if verif_dsic(m, true_types).0 && verif_ir(m, true_types).0 {
            n += 1;
        }
    }
    let verdict = if n == 0 {
        "VIDE - temoin VRAI (un agent de type 0 existe)".to_string()
    } else {
        format!("NON VIDE : {} survivant(s)", n)
    };
    println!("  true={} : admissibles = {}  -> {}", py_tuple(true_types), n, verdict);
}

// === Cote 2 : le contre-temoin sur (1,1) ===
let mut issue_s: BTreeMap<(i32, i32), i32> = BTreeMap::new();
let mut pay_s: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
for p in PROFILES.iter() {
    issue_s.insert(*p, 1);
    pay_s.insert(*p, (1, 1));
}
let m_survivant: Mec = (issue_s, pay_s);
println!();
println!("Le survivant uniforme sur (1,1), exhibe et verifie :");
println!("  non-déviation locale : {}   IR point-prior : {}",
    py_bool(verif_dsic(&m_survivant, TRUE_11).0), py_bool(verif_ir(&m_survivant, TRUE_11).0));
println!("  Bien-etre : {}   Revenu : {}  (extraction complete)",
    verif_welfare(&m_survivant, TRUE_11), verif_revenue(&m_survivant, TRUE_11));

// === Cote 3 : la ligne d'arrivee uniforme sur les types et reports ===
let cands = enumerer_tous();
let mut n_ex = 0i32;
let mut rev_ex: BTreeSet<i32> = BTreeSet::new();
let mut avec_paiement_non_nul = 0i32;
for m in cands.iter() {
    if verif_ir_uniforme(m) && verif_dsic(m, TRUE_11).0 {
        n_ex += 1;
        rev_ex.insert(verif_revenue(m, TRUE_11));
        if m.1.values().any(|v| *v != (0, 0)) {
            avec_paiement_non_nul += 1;
        }
    }
}
println!();
println!("Lecture uniforme sur les types et reports (celle de la preuve du temoin) sur (1,1) :");
println!("  admissibles non-déviation locale + IR uniforme sur les types et reports = {}, dont un paiement non nul = {}", n_ex, avec_paiement_non_nul);
let rev_texte: Vec<String> = rev_ex.iter().map(|v| v.to_string()).collect();
println!("  revenus possibles = [{}]", rev_texte.join(", "));
println!("  -> sous uniforme sur les types et reports, paiements ~ 0 est un THEOREME (theta=0 force p=0 partout)");

Temoin GT-16b (paiement uniforme >= 1), IR point-prior, instance par instance :


  true=(0, 0) : admissibles = 0  -> VIDE - temoin VRAI (un agent de type 0 existe)


  true=(0, 1) : admissibles = 0  -> VIDE - temoin VRAI (un agent de type 0 existe)


  true=(1, 0) : admissibles = 0  -> VIDE - temoin VRAI (un agent de type 0 existe)


  true=(1, 1) : admissibles = 1  -> NON VIDE : 1 survivant(s)


Le survivant uniforme sur (1,1), exhibe et verifie :


... (2 outputs left out)

Lecture uniforme sur les types et reports (celle de la preuve du temoin) sur (1,1) :


  admissibles non-déviation locale + IR uniforme sur les types et reports = 2, dont un paiement non nul = 0


  revenus possibles = [0]


  -> sous uniforme sur les types et reports, paiements ~ 0 est un THEOREME (theta=0 force p=0 partout)


**Lecture de la sortie.** La contrainte de paiement positif partout ne laisse aucun mécanisme admissible sur les trois instances contenant un type 0. Sur(1,1), le mécanisme uniforme issue 1/paiements(1,1) survit et reçoit un revenu 2. La conclusion d'ensemble vide doit donc être bornée à la contrainte de participation réellement employée.

Avec l'IR uniforme sur tous les types et reports, le type 0 est testé à chaque report : les paiements non négatifs sont alors tous nuls. Ce résultat provient du quantificateur supplémentaire, pas d'un calcul d'espérance ni d'un théorème extérieur au modèle.

## Exercice 3 : non-déviation locale stricte

Construisez les marges signées du vérificateur local : utilité au report courant moins utilité après changement unilatéral. Une marge positive favorise le report courant, une marge négative rend le changement profitable. Retournez les huit marges dans `marges_locales` / `margesLocales`, en parcourant les quatre profils puis les deux agents. Les utilités de ce domaine sont entières : les marges attendues le sont aussi.

Les trois fixtures contiennent des marges négatives, nulles et positives. Le garde vérifie ensuite chaque mécanisme admissible au bien-être maximal et revenu positif. Démontrez pourquoi le contrôle des deux sens d'une même arête empêche toutes les marges d'être strictement positives ; reliez cette preuve au comptage obtenu lorsque votre fonction est complétée. Le cas de l'extracteur et celui du mécanisme uniforme doivent être interprétés dans le modèle local, sans les confondre avec un test DSIC standard.

In [7]:
use std::collections::BTreeMap;

type Mec = (BTreeMap<(i32, i32), i32>, BTreeMap<(i32, i32), (i32, i32)>);

const PROFILES: [(i32, i32); 4] = [(0, 0), (0, 1), (1, 0), (1, 1)];   // profils de reports (2 agents, 2 types)
const N_AGENTS: usize = 2;

// Exercice 3 a completer
// 1. Definir verif_dsic_strict(M, true) : u_sincere > u_dev pour toute deviation
// 2. Re-enumerer l'espace complet sur (1,1) : compter les non-deviation locale-strict + IR a welfare optimal
// 3. Max revenu sous non-deviation locale strict vs 2 (faible) ; et le survivant uniforme de la section 4 ?
// Indice : sur ce domaine, les ecarts d'utilite sont entiers -- l'egalite exacte est le seul cas filtre
fn marges_locales(_mecanisme: &Mec, _type0: i32, _type1: i32) -> Option<[i64; 8]> {
    // TODO etudiant : huit marges, ordre profil puis agent
    None
}

let m_extracteur: Mec = (
    BTreeMap::from([((0, 0), 0), ((0, 1), 0), ((1, 0), 0), ((1, 1), 1)]),
    BTreeMap::from([((0, 0), (0, 0)), ((0, 1), (0, 0)), ((1, 0), (0, 0)), ((1, 1), (1, 1))]),
);
let mut issue_s: BTreeMap<(i32, i32), i32> = BTreeMap::new();
let mut pay_s: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
for p in PROFILES.iter() {
    issue_s.insert(*p, 1);
    pay_s.insert(*p, (1, 1));
}
let m_survivant: Mec = (issue_s, pay_s);
let mut sans_issue: BTreeMap<(i32, i32), i32> = BTreeMap::new();
let mut sans_pay: BTreeMap<(i32, i32), (i32, i32)> = BTreeMap::new();
for p in PROFILES.iter() {
    let issue = if *p == (1, 1) { 1 } else { 0 };
    sans_issue.insert(*p, issue);
    sans_pay.insert(*p, (0, 0));
}
let sans_paiement: Mec = (sans_issue, sans_pay);

let cas_marges: Vec<(Mec, [f64; 8])> = vec![
    (m_extracteur, [0.0; 8]),
    (m_survivant, [0.0; 8]),
    (sans_paiement, [0.0, 0.0, -1.0, 0.0, 0.0, -1.0, 1.0, 1.0]),
];
let mut reponses_marges = 0;
for (mecanisme, _attendu) in cas_marges.iter() {
    let rep = marges_locales(mecanisme, 1, 1);
    if rep.is_none() {
        continue;
    }
    reponses_marges += 1;
}
if reponses_marges == 0 {
    println!("Exercice 3 à compléter : non-déviation stricte locale");
} else {
    // La post-garde de la source (re-enumeration des 4096 candidats, assertion
    // marges == 0) depend de l'implementation du stub : elle est rejouee par l'etudiant.
    println!("Exercice 3 : marges validées ; post-garde à rejouer sur les 4096 candidats");
};
// Le point-virgule final supprime le `execute_result ()` qu'evcxr publie
// quand la derniere instruction d'une cellule est un bloc (`if`/`else`).

Exercice 3 à compléter : non-déviation stricte locale


## 5. Verdict — ce que l'effondrement coûte, ce qui survit

| Question | Réponse mesurée | Portée |
|---|---|---|
| Le générateur est-il admissible ? | Non : l'IR passe, mais la non-déviation locale échoue | Types fixés(1,1), table complète vérifiée |
| Son bien-être et son revenu ? | Bien-être2, revenu 0 | Il maximise seulement le premier critère |
| Que permet l'espace complet ? |47 admissibles, 34 au premier meilleur,29 à revenu positif, maximum2 | Instance(1,1), paiements binaires |
| Le paiement uniforme positif est-il impossible ? | Oui sur les trois instances avec un type 0 ; un survivant sur (1,1) | IR à types fixés |
| Que change l'IR uniforme ? | Tout paiement non nul est exclu | Tous types et tous reports sont testés |
| Revenu maximal au premier meilleur ? |0,1,1,2 dans l'ordre des quatre instances | Égal à la somme des types dans ce jouet |

Le même vérificateur révèle le défaut du générateur et certifie les mécanismes issus de l'énumération. Le critère optimisé et les contraintes doivent donc être contrôlés séparément, sur le livrable réel.

### Questions ouvertes — ce que ce jouet ne dit pas

- **Types distribués.** Comment changer le problème quand le designer ne connaît plus les types présents ? Il faut définir un prior, la participation et les incitations correspondantes avant de comparer les revenus ; cette extension n'est pas exécutée ici.
- **Coût de l'issue.** La provision de l'issue 1 est gratuite dans ce modèle. Ajouter un coût c exige de distinguer revenu, coût et surplus avant de chercher un mécanisme qui finance la provision.
- **Autre définition des incitations.** Comment remplacer la comparaison de tous les reports à types fixés par une comparaison entre report sincère et déviations pour chaque type ? Le résultat local d'impossibilité de la stricte positivité des deux sens ne doit pas être transféré sans refaire cette définition.

## Exercices — récapitulatif

Les trois activités réparties démontrent la borne IR des types nuls, mesurent la gamme de paiements étendue en conservant son budget, puis construisent les marges de non-déviation stricte. Rédigez les arguments qui relient vos sorties aux limites du modèle.

## À retenir

1. Le générateur visite 16 tables d'issues sur 4096 mécanismes. Il atteint le bien-être 2 et un revenu nul sur (1,1), mais sa table échoue au test local de non-déviation.
2. L'énumération complète trouve 47 admissibles, 34 au bien-être maximal et 29 à revenu positif sur (1,1). Le revenu maximal y vaut 2, atteint par l'extracteur ; sur les quatre instances il vaut 0,1,1,2.
3. L'extracteur a des utilités locales nulles pour les types fixés(1,1). Cela ne démontre pas une propriété générale de DSIC. La stricte positivité des marges dans les deux sens d'une arête est impossible.
4. L'IR à types fixés et l'IR uniforme sur tous les types/reports n'ont pas le même ensemble admissible. La seconde impose des paiements nuls ; ce n'est pas une IR calculée en espérance.

La méthode est de mesurer tout l'espace annoncé, d'employer un vérificateur indépendant et de nommer exactement les quantificateurs des contraintes. Les exercices gardent leurs fonctions à compléter pour vérifier les bornes, les budgets de l'énumération étendue et les marges signées.

## Vérifiez votre compréhension

1. Le générateur atteint J=2. Pourquoi cela ne suffit-il pas ?

<details><summary>Réponse</summary>Il optimise seulement le bien-être. Sa table et ses paiements nuls donnent bien J=2, mais le vérificateur local affiche une déviation profitable : pour les types (1,1), l'agent 0 gagne 1 en passant du report (0,1) au report (1,1). L'IR passe et le revenu reste nul ; ces trois tests répondent à des questions différentes.</details>

2. Que signifie l'indifférence de l'extracteur ?

<details><summary>Réponse</summary>Dans l'expérience aux types fixés(1,1), toute utilité locale vaut 0, aux quatre reports : les inégalités des deux sens sont donc des égalités. Le test ne prouve pas une incitation générale au report sincère ni une robustesse hors de ce modèle. Par exemple, si un agent était de type 0 et l'autre reportait1, passer de son report 0 à 1 le ferait payer 1 pour une valeur nulle : il perdrait 1, et non 0.</details>

3. Pourquoi le paiement uniforme survit-il sur (1,1) mais pas sous IR uniforme ?

<details><summary>Réponse</summary>À types fixés(1,1), issue 1 et paiement 1 donnent une utilité nulle à chacun. L'IR uniforme teste aussi le type 0 au même report ; son utilité serait−1, donc le mécanisme est rejeté. C'est un changement de quantificateur, sans moyenne sur une distribution.</details>

4. Le revenu maximal égal à la somme des types est-il un résultat général d'AMD ?

<details><summary>Réponse</summary>Non. Il est atteint ici dans un domaine binaire, aux types connus du designer et avec les contraintes locales exécutées. D'autres priors, coûts ou définitions des incitations constituent d'autres problèmes à définir et à résoudre.</details>

## Pour aller plus loin

**Référence pédagogique primaire.** Le calcul porté et relu est le [carnet Rust](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Rust.ipynb), portage du [source Python versionné à a8566836](https://github.com/jsboige/CoursIA/blob/a8566836bb4246cfedbf6923f63fb0b2ab37d368/MyIA.AI.Notebooks/GameTheory/GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb), avec ses énumérations et ses vérificateurs explicites. Les expériences du présent carnet vérifient ce modèle local, pas les théorèmes d’une publication externe.

**Contexte historique conservé du source.** Les notices ci-dessous étaient présentes dans ce source et situent le thème ; aucune nouvelle lecture de copie archivée ni vérification Crossref de première main n’est revendiquée par ce port. L’IR robuste de ce jouet ne reproduit pas l’IR ex-interim usuelle.

- Green, J., & Laffont, J.-J. (1977). Characterization of satisfactory mechanisms for the revelation of preferences for public goods. *Econometrica*, 45(2), 427. https://doi.org/10.2307/1911219
- Rochet, J.-C. (1987). A necessary and sufficient condition for rationalizability in a quasi-linear context. *Journal of Mathematical Economics*, 16(2), 191-200. https://doi.org/10.1016/0304-4068(87)90007-3
- Saijo, T. (1988). Strategy space reduction in Maskin's theorem: sufficient conditions for Nash implementation. *Econometrica*, 56(3), 693. https://doi.org/10.2307/1911706
- Myerson, R. B., & Satterthwaite, M. A. (1983). Efficient mechanisms for bilateral trading. *Journal of Economic Theory*, 29(2), 265-281. https://doi.org/10.1016/0022-0531(83)90048-0

**Carnet suivant.** [GameTheory-16d — Joueurs LLM (Othman-Sandholm)](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) -- du designer qui optimise dans un espace borné aux **joueurs** qui ne sont pas scriptés : le gel pré-enregistré et les contrôles scriptés y jouent le rôle que le vérificateur séparé joue ici.

**Carnets voisins.**

- [GameTheory-16b — Automated Mechanism Design](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) -- l'organe repris tel quel, et le générateur dont le carnet mesure l'effondrement.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : principe de révélation, VCG, et les limites que le théorème d'impossibilité pose à toute conception de mécanisme.
- [GameTheory-16k — Théorie des enchères (Myerson)](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb) -- le même Myerson, côté vendeur : la réserve qui retire la rente d'information.
